Member A:
I was responsible for developing and evaluating the recommendation component of HealthRank. The main tasks included implementing a popularity-based baseline, developing a Bayesian Personalised Ranking Matrix Factorisation (BPR-MF) model, extending the model with health-aware ranking, selecting the health-weight parameter using validation data, and evaluating the final models on the test set. A reusable Python inference interface was also implemented to facilitate integration with the rest of the application.

The main objective was to investigate whether incorporating meal health information into personalised recommendations could improve the average health quality of recommended meals while maintaining recommendation ranking performance.

In [ ]:
# Step 1: Inspect your data in Colab
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/HealthRank")
RAW_DIR = ROOT / "data" / "raw" / "MealRec+H"
PROCESSED_DIR = ROOT / "data" / "processed" / "MealRec+H"

print("=== RAW FILES ===")
for p in sorted(RAW_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(RAW_DIR)}  ({p.stat().st_size:,} bytes)")

print("\n=== PROCESSED FILES ===")
for p in sorted(PROCESSED_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(PROCESSED_DIR)}  ({p.stat().st_size:,} bytes)")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== RAW FILES ===
course_category.txt  (69,264 bytes)
healthiness/course_fsa.txt  (18,039 bytes)
healthiness/course_who.txt  (14,560 bytes)
healthiness/meal_fsa.txt  (51,620 bytes)
healthiness/meal_who.txt  (53,220 bytes)
healthiness/user_fsa.txt  (24,518 bytes)
healthiness/user_who.txt  (25,996 bytes)
meal_course.txt  (100,287 bytes)
meta_data/DateNote.txt  (271 bytes)
meta_data/course2index.txt  (80,565 bytes)
meta_data/user2index.txt  (18,841 bytes)
user_course.txt  (1,287,430 bytes)
user_meal_test.txt  (39,806 bytes)
user_meal_train.txt  (318,480 bytes)
user_meal_tune.txt  (39,857 bytes)

=== PROCESSED FILES ===
course_health.csv  (97,027 bytes)
meal_course.csv  (230,104 bytes)
meal_health.csv  (321,619 bytes)
stats.json  (664 bytes)
test.csv  (39,822 bytes)
train.csv  (318,496 bytes)
user_health_train.csv  (90,040 bytes)
val.csv  (39,873 bytes)


In [ ]:
# Step 2: Check interaction logs and health scores

import pandas as pd

def preview_file(path, n=5):
    print(f"\n{'=' * 70}\nFILE: {path.name}")
    try:
        df = pd.read_csv(
            path,
            sep=None,
            engine="python",
            nrows=n,
            encoding="utf-8"
        )
        print("Columns:", df.columns.tolist())
        print(df.to_string(index=False))
    except Exception as e:
        print("Could not parse automatically:", e)
        print(path.read_text(encoding="utf-8", errors="replace")[:1000])

for folder in [RAW_DIR, PROCESSED_DIR]:
    for p in sorted(folder.rglob("*")):
        if p.is_file() and p.suffix.lower() in {
            ".csv", ".tsv", ".txt"
        }:
            preview_file(p)



FILE: course_category.txt
Columns: ['851', '0']
 851  0
 604  1
 290  2
 653  0
 452  1
 101  2

FILE: course_fsa.txt
Columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2']
 Unnamed: 0  Unnamed: 1  Unnamed: 2
          9         NaN         NaN
          7         NaN         NaN
          9         NaN         NaN
          8         NaN         NaN
          9         NaN         NaN

FILE: course_who.txt
Columns: ['Unnamed: 0', 'Unnamed: 1']
 Unnamed: 0  Unnamed: 1
        2.0         NaN
        NaN         NaN
        3.0         NaN
        3.0         NaN
        3.0         NaN

FILE: meal_fsa.txt
Columns: ['10.', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', '4']
      10.  Unnamed: 1  Unnamed: 2  Unnamed: 3  Unnamed: 4  Unnamed: 5  Unnamed: 6  Unnamed: 7  Unnamed: 8  Unnamed: 9  Unnamed: 10  Unnamed: 11  Unnamed: 12  Unnamed: 13   4
10.0000

In [ ]:
# Step 3: First verify B's data interface
import os
import sys

ROOT = "/content/drive/MyDrive/HealthRank"
os.chdir(ROOT)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.dataset import load_data

data = load_data("MealRec+H")

for name in ["train", "val", "test"]:
    df = getattr(data, name)
    print(f"\n{name}:")
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    print(df.head())
    print("users:", df["user_id"].nunique())
    print("meals:", df["meal_id"].nunique())


train:
shape: (37413, 2)
columns: ['user_id', 'meal_id']
   user_id  meal_id
0      153       43
1        3     3151
2        3     1265
3      651      819
4     1016     2269
users: 1575
meals: 3810

val:
shape: (4677, 2)
columns: ['user_id', 'meal_id']
   user_id  meal_id
0      199      603
1      554     2919
2     1470      605
3      451      302
4      325     1523
users: 1131
meals: 2332

test:
shape: (4677, 2)
columns: ['user_id', 'meal_id']
   user_id  meal_id
0      811     1006
1       77     2865
2      983     3030
3      258     3227
4     1082     1626
users: 1114
meals: 2347


In [ ]:
# Step 4: Confirm the definition of health score
import pandas as pd
import json

base = "data/processed/MealRec+H"

health = pd.read_csv(f"{base}/meal_health.csv")
print(health.describe().to_string())
print("\nHealth score example:")
print(health.head(10).to_string(index=False))

with open(f"{base}/stats.json", "r") as f:
    print("\nData statistics:")
    print(json.dumps(json.load(f), indent=2))

           meal_id          fsa          who     fsa_norm     who_norm  health_score
count  3817.000000  3817.000000  3817.000000  3817.000000  3817.000000   3817.000000
mean   1908.000000     9.275784     2.060169     0.358632     0.345367      0.352000
std    1102.017317     0.859838     0.633353     0.128976     0.126671      0.106520
min       0.000000     5.000000     0.333333     0.000000     0.000000      0.083333
25%     954.000000     8.666667     1.666667     0.250000     0.266667      0.283333
50%    1908.000000     9.333333     2.000000     0.350000     0.333333      0.333333
75%    2862.000000    10.000000     2.333333     0.450000     0.400000      0.408333
max    3816.000000    11.666667     5.333333     1.000000     1.000000      0.900000

Health score example:
 meal_id       fsa      who  fsa_norm  who_norm  health_score
       0 10.333333 2.000000      0.20  0.333333      0.266667
       1 10.333333 1.666667      0.20  0.266667      0.233333
       2 10.333333 2.33333

Popularity：Ranked by the number of meal interactions in the training set.



BPR-MF：Learns personalized preferences using user and meal latent vectors.



Health-aware BPR：Incorporates meal health scores into the ranking learning process of BPR-MF.   

In [ ]:
# Step 5: Now create src/models.py

%%writefile /content/drive/MyDrive/HealthRank/src/models.py

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


class Popularity:
    """Rank meals by interaction frequency in the training set."""

    def fit(self, train):
        self.popularity = (
            train["meal_id"].value_counts().to_dict()
        )
        self.seen = (
            train.groupby("user_id")["meal_id"]
            .apply(set).to_dict()
        )
        self.all_meals = sorted(self.popularity.keys())
        return self

    def recommend(self, user_id, k=10):
        seen = self.seen.get(user_id, set())
        candidates = [
            m for m in self.all_meals if m not in seen
        ]
        candidates.sort(
            key=lambda m: self.popularity[m], reverse=True
        )
        return pd.DataFrame({
            "meal_id": candidates[:k],
            "score": [
                float(self.popularity[m]) for m in candidates[:k]
            ]
        })


class BPRMF(nn.Module):
    """
    BPR matrix factorization.
    lambda_health=0: ordinary BPR-MF.
    lambda_health>0: health-aware pairwise ranking.
    """

    def __init__(
        self,
        n_users,
        n_meals,
        health_scores=None,
        factors=64,
        lambda_health=0.0,
        reg=1e-4
    ):
        super().__init__()

        self.n_users = n_users
        self.n_meals = n_meals
        self.lambda_health = lambda_health
        self.reg = reg

        self.user_embedding = nn.Embedding(n_users, factors)
        self.meal_embedding = nn.Embedding(n_meals, factors)

        nn.init.normal_(self.user_embedding.weight, std=0.05)
        nn.init.normal_(self.meal_embedding.weight, std=0.05)

        if health_scores is None:
            health_scores = np.zeros(n_meals, dtype=np.float32)

        self.register_buffer(
            "health",
            torch.tensor(health_scores, dtype=torch.float32)
        )

    def forward(self, users, meals):
        u = self.user_embedding(users)
        m = self.meal_embedding(meals)
        preference = (u * m).sum(dim=1)

        return preference + self.lambda_health * self.health[meals]

    def bpr_loss(self, users, pos_meals, neg_meals):
        pos_scores = self(users, pos_meals)
        neg_scores = self(users, neg_meals)

        ranking_loss = -F.logsigmoid(
            pos_scores - neg_scores
        ).mean()

        u = self.user_embedding(users)
        p = self.meal_embedding(pos_meals)
        n = self.meal_embedding(neg_meals)

        regularization = self.reg * (
            u.pow(2).sum(dim=1)
            + p.pow(2).sum(dim=1)
            + n.pow(2).sum(dim=1)
        ).mean()

        return ranking_loss + regularization


def build_health_array(meal_health, n_meals):
    """
    Convert meal_health.csv into an array indexed by meal_id.
    Scale scores to [0, 1] while preserving their direction.
    """
    scores = np.zeros(n_meals, dtype=np.float32)

    valid = meal_health.dropna(
        subset=["meal_id", "health_score"]
    )
    ids = valid["meal_id"].astype(int).to_numpy()
    values = valid["health_score"].astype(float).to_numpy()

    low, high = values.min(), values.max()
    if high > low:
        values = (values - low) / (high - low)
    else:
        values = np.zeros_like(values)

    mask = (ids >= 0) & (ids < n_meals)
    scores[ids[mask]] = values[mask]

    return scores


def train_bpr(
    train,
    n_users,
    n_meals,
    health_scores=None,
    lambda_health=0.0,
    factors=64,
    epochs=20,
    batch_size=2048,
    lr=1e-3,
    reg=1e-4,
    seed=42,
    device=None
):
    """Train BPR-MF using uniform negative sampling."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    device = device or (
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    users = train["user_id"].to_numpy(dtype=np.int64)
    positives = train["meal_id"].to_numpy(dtype=np.int64)

    # Positive meals for each user; avoid sampling known positives.
    seen = {}
    for u, m in zip(users, positives):
        seen.setdefault(int(u), set()).add(int(m))

    model = BPRMF(
        n_users=n_users,
        n_meals=n_meals,
        health_scores=health_scores,
        factors=factors,
        lambda_health=lambda_health,
        reg=reg
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    n = len(train)
    all_meals = np.arange(n_meals)

    for epoch in range(epochs):
        order = np.random.permutation(n)
        total_loss = 0.0
        total_count = 0

        model.train()
        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]
            u_np = users[idx]
            p_np = positives[idx]

            neg_np = np.random.randint(
                0, n_meals, size=len(idx)
            )

            # Resample negatives that are known positives.
            for j, u in enumerate(u_np):
                while int(neg_np[j]) in seen[int(u)]:
                    neg_np[j] = np.random.randint(0, n_meals)

            u = torch.as_tensor(u_np, device=device)
            p = torch.as_tensor(p_np, device=device)
            neg = torch.as_tensor(neg_np, device=device)

            optimizer.zero_grad()
            loss = model.bpr_loss(u, p, neg)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(idx)
            total_count += len(idx)

        print(
            f"Epoch {epoch + 1:02d}/{epochs} "
            f"loss={total_loss / total_count:.4f}"
        )

    model.eval()
    model.seen = seen
    model.device_name = device
    return model


def recommend_bpr(model, user_id, k=10, chunk_size=1024):
    """Return top-k unseen meals for a user."""
    if user_id < 0 or user_id >= model.n_users:
        raise ValueError(f"Unknown user_id: {user_id}")

    seen = model.seen.get(int(user_id), set())
    device = model.device_name

    model.eval()
    score_chunks = []

    with torch.no_grad():
        for start in range(0, model.n_meals, chunk_size):
            end = min(start + chunk_size, model.n_meals)
            meals = torch.arange(start, end, device=device)
            users = torch.full_like(meals, int(user_id))
            scores = model(users, meals)
            score_chunks.append(scores.cpu().numpy())

    scores = np.concatenate(score_chunks)

    if seen:
        seen_ids = np.fromiter(seen, dtype=np.int64)
        scores[seen_ids] = -np.inf

    k = min(k, int(np.isfinite(scores).sum()))
    if k == 0:
        return pd.DataFrame(columns=["meal_id", "score"])

    top_ids = np.argpartition(-scores, k - 1)[:k]
    top_ids = top_ids[np.argsort(-scores[top_ids])]

    return pd.DataFrame({
        "meal_id": top_ids,
        "score": scores[top_ids].astype(float)
    })


Overwriting /content/drive/MyDrive/HealthRank/src/models.py


In [ ]:
# Step 6: Run the first round of model experiments

import json
import pandas as pd
import numpy as np
import torch

from src.dataset import load_data
from src.models import (
    Popularity,
    build_health_array,
    train_bpr,
    recommend_bpr
)

data = load_data("MealRec+H")

BASE = "data/processed/MealRec+H"
health_df = pd.read_csv(f"{BASE}/meal_health.csv")

with open(f"{BASE}/stats.json") as f:
    stats = json.load(f)

n_users = stats["n_users"]
n_meals = stats["n_meals"]

health_scores = build_health_array(health_df, n_meals)

print("Users:", n_users)
print("Meals:", n_meals)
print("Health array:", health_scores.shape)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

# Baseline 1: Popularity
pop_model = Popularity().fit(data.train)
print("\nPopularity example:")
print(pop_model.recommend(user_id=3, k=10))

# Baseline 2: BPR-MF
bpr_model = train_bpr(
    train=data.train,
    n_users=n_users,
    n_meals=n_meals,
    health_scores=health_scores,
    lambda_health=0.0,
    factors=64,
    epochs=20
)

print("\nBPR-MF example:")
print(recommend_bpr(bpr_model, user_id=3, k=10))

# Candidate method: health-aware BPR
health_model = train_bpr(
    train=data.train,
    n_users=n_users,
    n_meals=n_meals,
    health_scores=health_scores,
    lambda_health=0.1,
    factors=64,
    epochs=20
)

print("\nHealth-aware BPR example:")
print(recommend_bpr(health_model, user_id=3, k=10))


Users: 1575
Meals: 3817
Health array: (3817,)
Device: cuda

Popularity example:
   meal_id  score
0      391   86.0
1      957   77.0
2       20   67.0
3      216   66.0
4       75   65.0
5     1111   65.0
6      694   63.0
7      894   60.0
8      376   59.0
9      281   55.0
Epoch 01/20 loss=0.6933
Epoch 02/20 loss=0.6905
Epoch 03/20 loss=0.6882
Epoch 04/20 loss=0.6857
Epoch 05/20 loss=0.6830
Epoch 06/20 loss=0.6797
Epoch 07/20 loss=0.6758
Epoch 08/20 loss=0.6712
Epoch 09/20 loss=0.6656
Epoch 10/20 loss=0.6589
Epoch 11/20 loss=0.6509
Epoch 12/20 loss=0.6416
Epoch 13/20 loss=0.6305
Epoch 14/20 loss=0.6181
Epoch 15/20 loss=0.6039
Epoch 16/20 loss=0.5875
Epoch 17/20 loss=0.5709
Epoch 18/20 loss=0.5526
Epoch 19/20 loss=0.5343
Epoch 20/20 loss=0.5145

BPR-MF example:
   meal_id     score
0      608  1.174648
1      486  1.119356
2       73  1.086860
3      588  1.075331
4     1757  1.041720
5      129  0.972742
6     2079  0.966115
7     1173  0.893376
8     2635  0.857771
9     1571  0.8

In [ ]:
# Step 7: Compare the three models using the validation set

import numpy as np
import pandas as pd
import torch

K = 10

# Build full user-meal score matrices.
# This makes evaluation use the same candidate meal universe.
def get_score_matrix(model, n_users, n_meals, popularity=None):
    if popularity is not None:
        scores = np.zeros(n_meals, dtype=np.float32)
        for meal_id, count in popularity.popularity.items():
            scores[int(meal_id)] = float(count)
        return np.tile(scores, (n_users, 1))

    model.eval()
    with torch.no_grad():
        user_emb = model.user_embedding.weight
        meal_emb = model.meal_embedding.weight
        matrix = user_emb @ meal_emb.T

        if model.lambda_health != 0:
            matrix = matrix + (
                model.lambda_health * model.health.unsqueeze(0)
            )

        return matrix.cpu().numpy()


def evaluate_topk(score_matrix, train, val, health_df, k=10):
    n_users, n_meals = score_matrix.shape

    seen = train.groupby("user_id")["meal_id"].apply(
        lambda x: set(x.astype(int))
    ).to_dict()

    truth = val.groupby("user_id")["meal_id"].apply(
        lambda x: set(x.astype(int))
    ).to_dict()

    health_map = health_df.set_index("meal_id")[
        "health_score"
    ].to_dict()

    recalls, precisions, ndcgs, health_values = [], [], [], []

    for user_id, relevant in truth.items():
        user_id = int(user_id)
        relevant = set(map(int, relevant))

        scores = score_matrix[user_id].copy()

        # Do not recommend meals already observed in training.
        if user_id in seen:
            scores[list(seen[user_id])] = -np.inf

        top = np.argsort(-scores)[:k]
        hits = np.array([int(m in relevant) for m in top])

        recalls.append(
            hits.sum() / len(relevant) if relevant else 0.0
        )
        precisions.append(hits.sum() / k)

        discounts = 1.0 / np.log2(np.arange(2, k + 2))
        dcg = np.sum(hits * discounts)

        ideal_hits = min(len(relevant), k)
        idcg = np.sum(
            1.0 / np.log2(np.arange(2, ideal_hits + 2))
        ) if ideal_hits > 0 else 0.0

        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

        meal_health = [
            health_map.get(int(m), np.nan) for m in top
        ]
        health_values.append(np.nanmean(meal_health))

    return {
        f"Recall@{k}": np.mean(recalls),
        f"Precision@{k}": np.mean(precisions),
        f"NDCG@{k}": np.mean(ndcgs),
        f"AvgHealth@{k}": np.nanmean(health_values)
    }


pop_scores = get_score_matrix(
    pop_model, n_users, n_meals, popularity=pop_model
)
bpr_scores = get_score_matrix(
    bpr_model, n_users, n_meals
)
health_scores_matrix = get_score_matrix(
    health_model, n_users, n_meals
)

results = []
for name, matrix in [
    ("Popularity", pop_scores),
    ("BPR-MF", bpr_scores),
    ("Health-aware BPR", health_scores_matrix)
]:
    row = {"Model": name}
    row.update(
        evaluate_topk(
            matrix, data.train, data.val, health_df, k=K
        )
    )
    results.append(row)

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


           Model  Recall@10  Precision@10  NDCG@10  AvgHealth@10
      Popularity   0.025642      0.008576 0.018586      0.331142
          BPR-MF   0.043813      0.014500 0.033427      0.331852
Health-aware BPR   0.041413      0.014058 0.032484      0.353969


In [ ]:
# Step 8: Tune the health weight lambda_health
# lambda in {0,0.025,0.05,0.1,0.2,0.4}

# Coarse hyperparameter search on the validation set.
# Use the same training configuration and random seed
# for a fair comparison across lambda values.

lambda_grid = [0.0, 0.025, 0.05, 0.1, 0.2, 0.4]
tuning_results = []

for lam in lambda_grid:
    print(f"\n{'=' * 50}")
    print(f"Training lambda_health={lam}")

    model = train_bpr(
        train=data.train,
        n_users=n_users,
        n_meals=n_meals,
        health_scores=health_scores,
        lambda_health=lam,
        factors=64,
        epochs=10,       # coarse screening
        batch_size=2048,
        lr=1e-3,
        reg=1e-4,
        seed=42
    )

    score_matrix = get_score_matrix(model, n_users, n_meals)

    metrics = evaluate_topk(
        score_matrix,
        data.train,
        data.val,
        health_df,
        k=10
    )

    tuning_results.append({
        "lambda_health": lam,
        **metrics
    })

tuning_df = pd.DataFrame(tuning_results)
print("\nValidation tuning results:")
print(
    tuning_df.sort_values("NDCG@10", ascending=False)
    .to_string(index=False)
)



Training lambda_health=0.0
Epoch 01/10 loss=0.6933
Epoch 02/10 loss=0.6905
Epoch 03/10 loss=0.6882
Epoch 04/10 loss=0.6857
Epoch 05/10 loss=0.6830
Epoch 06/10 loss=0.6797
Epoch 07/10 loss=0.6758
Epoch 08/10 loss=0.6712
Epoch 09/10 loss=0.6656
Epoch 10/10 loss=0.6589

Training lambda_health=0.025
Epoch 01/10 loss=0.6934
Epoch 02/10 loss=0.6906
Epoch 03/10 loss=0.6883
Epoch 04/10 loss=0.6858
Epoch 05/10 loss=0.6831
Epoch 06/10 loss=0.6798
Epoch 07/10 loss=0.6759
Epoch 08/10 loss=0.6713
Epoch 09/10 loss=0.6657
Epoch 10/10 loss=0.6590

Training lambda_health=0.05
Epoch 01/10 loss=0.6935
Epoch 02/10 loss=0.6907
Epoch 03/10 loss=0.6884
Epoch 04/10 loss=0.6859
Epoch 05/10 loss=0.6832
Epoch 06/10 loss=0.6799
Epoch 07/10 loss=0.6760
Epoch 08/10 loss=0.6714
Epoch 09/10 loss=0.6658
Epoch 10/10 loss=0.6591

Training lambda_health=0.1
Epoch 01/10 loss=0.6938
Epoch 02/10 loss=0.6910
Epoch 03/10 loss=0.6887
Epoch 04/10 loss=0.6861
Epoch 05/10 loss=0.6835
Epoch 06/10 loss=0.6802
Epoch 07/10 loss=0.67

In [ ]:
# Step 9: Tune the health weight lambda_health again
# lambda in {0,0.005,0.01,0.015,0.025,0.04}

lambda_grid = [0.0, 0.005, 0.01, 0.015, 0.025, 0.04]
tuning_results = []

for lam in lambda_grid:
    print(f"\n{'=' * 50}\nTraining lambda_health={lam}")

    model = train_bpr(
        train=data.train,
        n_users=n_users,
        n_meals=n_meals,
        health_scores=health_scores,
        lambda_health=lam,
        factors=64,
        epochs=20,
        batch_size=2048,
        lr=1e-3,
        reg=1e-4,
        seed=42
    )

    score_matrix = get_score_matrix(model, n_users, n_meals)

    metrics = evaluate_topk(
        score_matrix,
        data.train,
        data.val,
        health_df,
        k=10
    )

    tuning_results.append({
        "lambda_health": lam,
        **metrics
    })

tuning_df = pd.DataFrame(tuning_results)

print("\nSorted by NDCG@10:")
print(
    tuning_df.sort_values(
        "NDCG@10", ascending=False
    ).to_string(index=False)
)

print("\nSorted by AvgHealth@10:")
print(
    tuning_df.sort_values(
        "AvgHealth@10", ascending=False
    ).to_string(index=False)
)



Training lambda_health=0.0
Epoch 01/20 loss=0.6933
Epoch 02/20 loss=0.6905
Epoch 03/20 loss=0.6882
Epoch 04/20 loss=0.6857
Epoch 05/20 loss=0.6830
Epoch 06/20 loss=0.6797
Epoch 07/20 loss=0.6758
Epoch 08/20 loss=0.6712
Epoch 09/20 loss=0.6656
Epoch 10/20 loss=0.6589
Epoch 11/20 loss=0.6509
Epoch 12/20 loss=0.6416
Epoch 13/20 loss=0.6305
Epoch 14/20 loss=0.6181
Epoch 15/20 loss=0.6039
Epoch 16/20 loss=0.5875
Epoch 17/20 loss=0.5709
Epoch 18/20 loss=0.5526
Epoch 19/20 loss=0.5343
Epoch 20/20 loss=0.5145

Training lambda_health=0.005
Epoch 01/20 loss=0.6933
Epoch 02/20 loss=0.6906
Epoch 03/20 loss=0.6883
Epoch 04/20 loss=0.6857
Epoch 05/20 loss=0.6830
Epoch 06/20 loss=0.6797
Epoch 07/20 loss=0.6758
Epoch 08/20 loss=0.6712
Epoch 09/20 loss=0.6656
Epoch 10/20 loss=0.6589
Epoch 11/20 loss=0.6509
Epoch 12/20 loss=0.6416
Epoch 13/20 loss=0.6305
Epoch 14/20 loss=0.6182
Epoch 15/20 loss=0.6039
Epoch 16/20 loss=0.5875
Epoch 17/20 loss=0.5709
Epoch 18/20 loss=0.5526
Epoch 19/20 loss=0.5343
Epoch 

In [ ]:
# Step 10: One place requires special checking
# The current evaluation code accesses the matrix via scores[user_id] and scores[meal_id]. This requires user_id and meal_id to be mapped to zero-based contiguous integers.
print("User ID range:",
      data.train["user_id"].min(),
      data.train["user_id"].max(),
      "unique:", data.train["user_id"].nunique())

print("Meal ID range:",
      data.train["meal_id"].min(),
      data.train["meal_id"].max(),
      "unique:", data.train["meal_id"].nunique())

print("Configured n_users:", n_users)
print("Configured n_meals:", n_meals)

User ID range: 0 1574 unique: 1575
Meal ID range: 0 3816 unique: 3810
Configured n_users: 1575
Configured n_meals: 3817


In [ ]:
# Step 11: Check the 7 meal IDs that do not appear in the training set
train_meals = set(data.train["meal_id"].astype(int))
val_meals = set(data.val["meal_id"].astype(int))
test_meals = set(data.test["meal_id"].astype(int))
all_meals = set(range(n_meals))

missing_train = all_meals - train_meals

print("Meals absent from train:", len(missing_train))
print("Also appearing in validation:",
      len(missing_train & val_meals))
print("Also appearing in test:",
      len(missing_train & test_meals))
print("Absent from all splits:",
      len(missing_train - val_meals - test_meals))

Meals absent from train: 7
Also appearing in validation: 6
Also appearing in test: 3
Absent from all splits: 0


In [ ]:
#Step 12: It is recommended to check the stability across different random seeds before finalizing the parameters.

seeds = [42, 123, 2026]
lambda_values = [0.0, 0.025]
seed_results = []

for seed in seeds:
    for lam in lambda_values:
        print(f"\nSeed={seed}, lambda_health={lam}")

        model = train_bpr(
            train=data.train,
            n_users=n_users,
            n_meals=n_meals,
            health_scores=health_scores,
            lambda_health=lam,
            factors=64,
            epochs=20,
            batch_size=2048,
            lr=1e-3,
            reg=1e-4,
            seed=seed
        )

        score_matrix = get_score_matrix(
            model, n_users, n_meals
        )

        metrics = evaluate_topk(
            score_matrix,
            data.train,
            data.val,
            health_df,
            k=10
        )

        seed_results.append({
            "seed": seed,
            "lambda_health": lam,
            **metrics
        })

seed_df = pd.DataFrame(seed_results)

print("\nAll results:")
print(seed_df.to_string(index=False))

print("\nMean and standard deviation:")
print(
    seed_df.groupby("lambda_health")[
        ["Recall@10", "Precision@10",
         "NDCG@10", "AvgHealth@10"]
    ].agg(["mean", "std"]).round(6)
)



Seed=42, lambda_health=0.0
Epoch 01/20 loss=0.6933
Epoch 02/20 loss=0.6905
Epoch 03/20 loss=0.6882
Epoch 04/20 loss=0.6857
Epoch 05/20 loss=0.6830
Epoch 06/20 loss=0.6797
Epoch 07/20 loss=0.6758
Epoch 08/20 loss=0.6712
Epoch 09/20 loss=0.6656
Epoch 10/20 loss=0.6589
Epoch 11/20 loss=0.6509
Epoch 12/20 loss=0.6416
Epoch 13/20 loss=0.6305
Epoch 14/20 loss=0.6181
Epoch 15/20 loss=0.6039
Epoch 16/20 loss=0.5875
Epoch 17/20 loss=0.5709
Epoch 18/20 loss=0.5526
Epoch 19/20 loss=0.5343
Epoch 20/20 loss=0.5145

Seed=42, lambda_health=0.025
Epoch 01/20 loss=0.6934
Epoch 02/20 loss=0.6906
Epoch 03/20 loss=0.6883
Epoch 04/20 loss=0.6858
Epoch 05/20 loss=0.6831
Epoch 06/20 loss=0.6798
Epoch 07/20 loss=0.6759
Epoch 08/20 loss=0.6713
Epoch 09/20 loss=0.6657
Epoch 10/20 loss=0.6590
Epoch 11/20 loss=0.6510
Epoch 12/20 loss=0.6417
Epoch 13/20 loss=0.6305
Epoch 14/20 loss=0.6182
Epoch 15/20 loss=0.6040
Epoch 16/20 loss=0.5876
Epoch 17/20 loss=0.5710
Epoch 18/20 loss=0.5526
Epoch 19/20 loss=0.5343
Epoch 

Keep λ=0.025 as a candidate parameter for the health-aware model, but do not claim it is significantly better than standard BPR-MF.

In [ ]:
# Step 13: Check the implementation of train_bpr()
import inspect
from src.models import train_bpr, BPRMF

print("=" * 20, "train_bpr()", "=" * 20)
print(inspect.getsource(train_bpr))

print("\n" + "=" * 20, "BPRMF", "=" * 20)
print(inspect.getsource(BPRMF))


==================== train_bpr() ====================
def train_bpr(
    train,
    n_users,
    n_meals,
    health_scores=None,
    lambda_health=0.0,
    factors=64,
    epochs=20,
    batch_size=2048,
    lr=1e-3,
    reg=1e-4,
    seed=42,
    device=None
):
    """Train BPR-MF using uniform negative sampling."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    device = device or (
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    users = train["user_id"].to_numpy(dtype=np.int64)
    positives = train["meal_id"].to_numpy(dtype=np.int64)

    # Positive meals for each user; avoid sampling known positives.
    seen = {}
    for u, m in zip(users, positives):
        seen.setdefault(int(u), set()).add(int(m))

    model = BPRMF(
        n_users=n_users,
        n_meals=n_meals,
        health_scores=health_scores,
        factors=factors,
        lambda_health=lambda_health,
        reg=reg
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters()

In [ ]:
# Step 14: Check if the data splits are consistent

# Check the basic statistics of the three datasets.
for name, df in [
    ("train", data.train),
    ("validation", data.val),
    ("test", data.test)
]:
    print(
        f"{name}: rows={len(df)}, "
        f"users={df['user_id'].nunique()}, "
        f"meals={df['meal_id'].nunique()}"
    )

# Check for duplicate user-meal interactions.
for name, df in [
    ("train", data.train),
    ("validation", data.val),
    ("test", data.test)
]:
    duplicates = df.duplicated(["user_id", "meal_id"]).sum()
    print(f"{name} duplicate interactions: {duplicates}")

# Check if there are duplicate interactions between the training set and the validation or test sets.
train_pairs = set(
    zip(data.train["user_id"], data.train["meal_id"])
)
val_pairs = set(
    zip(data.val["user_id"], data.val["meal_id"])
)
test_pairs = set(
    zip(data.test["user_id"], data.test["meal_id"])
)

print("Train-Val overlap:", len(train_pairs & val_pairs))
print("Train-Test overlap:", len(train_pairs & test_pairs))
print("Val-Test overlap:", len(val_pairs & test_pairs))


train: rows=37413, users=1575, meals=3810
validation: rows=4677, users=1131, meals=2332
test: rows=4677, users=1114, meals=2347
train duplicate interactions: 0
validation duplicate interactions: 0
test duplicate interactions: 0
Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


In [ ]:
# Step 15: Add the Popularity baseline
import inspect
from src.models import Popularity

print(inspect.getsource(Popularity))


class Popularity:
    """Rank meals by interaction frequency in the training set."""

    def fit(self, train):
        self.popularity = (
            train["meal_id"].value_counts().to_dict()
        )
        self.seen = (
            train.groupby("user_id")["meal_id"]
            .apply(set).to_dict()
        )
        self.all_meals = sorted(self.popularity.keys())
        return self

    def recommend(self, user_id, k=10):
        seen = self.seen.get(user_id, set())
        candidates = [
            m for m in self.all_meals if m not in seen
        ]
        candidates.sort(
            key=lambda m: self.popularity[m], reverse=True
        )
        return pd.DataFrame({
            "meal_id": candidates[:k],
            "score": [
                float(self.popularity[m]) for m in candidates[:k]
            ]
        })



In [ ]:
print("n_users =", n_users)
print("n_meals =", n_meals)
print("pop_scores shape =", pop_scores.shape)
print("max meal_id =", max(pop_model.popularity.keys()))

n_users = 1575
n_meals = 3817
pop_scores shape = (1575, 3817)
max meal_id = 3816


In [ ]:
#Step 16: Run the validation set evaluation for Popularity first

# 1. Fit Popularity using training data only

pop_model = Popularity().fit(data.train)

pop_scores = np.zeros(
    (n_users, n_meals),
    dtype=np.float32
)

for meal_id, count in pop_model.popularity.items():
    pop_scores[:, int(meal_id)] = float(count)

pop_metrics = evaluate_topk(
    score_matrix=pop_scores,
    train=data.train,
    val=data.val,
    health_df=health_df,
    k=10
)

print("Popularity Validation Results")
for metric, value in pop_metrics.items():
    print(f"{metric}: {value:.6f}")



Popularity Validation Results
Recall@10: 0.025642
Precision@10: 0.008576
NDCG@10: 0.018586
AvgHealth@10: 0.331142


In [ ]:
#Step 17: Put all three models into a single table
# Existing three-seed results from your previous experiment
summary = seed_df.groupby("lambda_health")[
    ["Recall@10", "Precision@10", "NDCG@10", "AvgHealth@10"]
].agg(["mean", "std"])

# Extract the mean results for standard BPR and health-aware BPR
comparison_rows = [
    {
        "Model": "Popularity",
        **pop_metrics
    },
    {
        "Model": "BPR-MF (lambda=0)",
        **summary.loc[0.0].xs("mean", level=1).to_dict()
    },
    {
        "Model": "Health-aware BPR (lambda=0.025)",
        **summary.loc[0.025].xs("mean", level=1).to_dict()
    }
]

comparison_df = pd.DataFrame(comparison_rows)

print(comparison_df.round(6).to_string(index=False))



                          Model  Recall@10  Precision@10  NDCG@10  AvgHealth@10
                     Popularity   0.025642      0.008576 0.018586      0.331142
              BPR-MF (lambda=0)   0.050502      0.015149 0.035581      0.331999
Health-aware BPR (lambda=0.025)   0.050244      0.015119 0.035601      0.336931


In [ ]:
#Step 18: Evaluate the test results for Popularity first
pop_test_metrics = evaluate_topk(
    score_matrix=pop_scores,
    train=data.train,
    val=data.test,
    health_df=health_df,
    k=10
)

print("Popularity Test Results")
for metric, value in pop_test_metrics.items():
    print(f"{metric}: {value:.6f}")

Popularity Test Results
Recall@10: 0.022604
Precision@10: 0.006732
NDCG@10: 0.013881
AvgHealth@10: 0.331120


In [ ]:
#Step 19: Retrain the two BPR models and evaluate them on the test set
# Final test evaluation: fixed hyperparameters selected using validation
seeds = [42, 123, 2026]
lambdas = [0.0, 0.025]

test_rows = []

for seed in seeds:
    for lam in lambdas:
        print(f"Training BPR: seed={seed}, lambda_health={lam}")

        model = train_bpr(
            train=data.train,
            n_users=n_users,
            n_meals=n_meals,
            health_scores=health_scores,
            lambda_health=lam,
            factors=64,
            epochs=20,
            batch_size=2048,
            lr=1e-3,
            reg=1e-4,
            seed=seed
        )

        scores = get_score_matrix(
            model=model,
            n_users=n_users,
            n_meals=n_meals
        )

        metrics = evaluate_topk(
            score_matrix=scores,
            train=data.train,
            val=data.test,
            health_df=health_df,
            k=10
        )

        test_rows.append({
            "seed": seed,
            "lambda_health": lam,
            **metrics
        })

test_df = pd.DataFrame(test_rows)

print("\nIndividual Test Results:")
print(test_df.round(6).to_string(index=False))

print("\nMean and Standard Deviation:")
test_summary = test_df.groupby("lambda_health")[
    ["Recall@10", "Precision@10", "NDCG@10", "AvgHealth@10"]
].agg(["mean", "std"])

print(test_summary.round(6))

Training BPR: seed=42, lambda_health=0.0
Epoch 01/20 loss=0.6933
Epoch 02/20 loss=0.6905
Epoch 03/20 loss=0.6882
Epoch 04/20 loss=0.6857
Epoch 05/20 loss=0.6830
Epoch 06/20 loss=0.6797
Epoch 07/20 loss=0.6758
Epoch 08/20 loss=0.6712
Epoch 09/20 loss=0.6656
Epoch 10/20 loss=0.6589
Epoch 11/20 loss=0.6509
Epoch 12/20 loss=0.6416
Epoch 13/20 loss=0.6305
Epoch 14/20 loss=0.6181
Epoch 15/20 loss=0.6039
Epoch 16/20 loss=0.5875
Epoch 17/20 loss=0.5709
Epoch 18/20 loss=0.5526
Epoch 19/20 loss=0.5343
Epoch 20/20 loss=0.5145
Training BPR: seed=42, lambda_health=0.025
Epoch 01/20 loss=0.6934
Epoch 02/20 loss=0.6906
Epoch 03/20 loss=0.6883
Epoch 04/20 loss=0.6858
Epoch 05/20 loss=0.6831
Epoch 06/20 loss=0.6798
Epoch 07/20 loss=0.6759
Epoch 08/20 loss=0.6713
Epoch 09/20 loss=0.6657
Epoch 10/20 loss=0.6590
Epoch 11/20 loss=0.6510
Epoch 12/20 loss=0.6417
Epoch 13/20 loss=0.6305
Epoch 14/20 loss=0.6182
Epoch 15/20 loss=0.6040
Epoch 16/20 loss=0.5876
Epoch 17/20 loss=0.5710
Epoch 18/20 loss=0.5526
Epoc

In [ ]:
# Step 20: Final test results
test_mean = test_df.groupby("lambda_health")[
    ["Recall@10", "Precision@10", "NDCG@10", "AvgHealth@10"]
].mean()

final_test_rows = [
    {
        "Model": "Popularity",
        **pop_test_metrics
    },
    {
        "Model": "BPR-MF (lambda=0)",
        **test_mean.loc[0.0].to_dict()
    },
    {
        "Model": "Health-aware BPR (lambda=0.025)",
        **test_mean.loc[0.025].to_dict()
    }
]

final_test_df = pd.DataFrame(final_test_rows)

print(final_test_df.round(6).to_string(index=False))

                          Model  Recall@10  Precision@10  NDCG@10  AvgHealth@10
                     Popularity   0.022604      0.006732 0.013881      0.331120
              BPR-MF (lambda=0)   0.049969      0.014811 0.035543      0.332133
Health-aware BPR (lambda=0.025)   0.050185      0.014901 0.035413      0.337029


In [ ]:
#Step 21: Save the results
import os
import json

RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Save individual results for all six BPR experiments
test_df.to_csv(
    os.path.join(RESULTS_DIR, "bpr_test_results_by_seed.csv"),
    index=False
)

# Save final comparison table
final_test_df.to_csv(
    os.path.join(RESULTS_DIR, "final_test_comparison.csv"),
    index=False
)

# Save validation comparison table, if available
if "comparison_df" in globals():
    comparison_df.to_csv(
        os.path.join(RESULTS_DIR, "validation_comparison.csv"),
        index=False
    )

# Save experiment configuration and metric definitions
config = {
    "dataset": "MealRec+H",
    "models": ["Popularity", "BPR-MF", "Health-aware BPR"],
    "seeds": [42, 123, 2026],
    "factors": 64,
    "epochs": 20,
    "batch_size": 2048,
    "learning_rate": 0.001,
    "regularization": 0.0001,
    "health_lambda": 0.025,
    "evaluation_k": 10,
    "selection_basis": "Validation set",
    "final_evaluation": "Test set"
}

with open(
    os.path.join(RESULTS_DIR, "experiment_config.json"),
    "w"
) as f:
    json.dump(config, f, indent=2)

print("Saved experiment results to:", RESULTS_DIR)
print(os.listdir(RESULTS_DIR))

Saved experiment results to: /content/drive/MyDrive/HealthRank/results
['dataset_stats_table.csv', 'validation_comparison.csv', 'experiment_config.json', 'bpr_test_mean_std.csv', 'final_test_comparison.csv', 'bpr_test_results_by_seed.csv']


In [ ]:
#Step 22: Check meal coverage in the training set
train_meals = set(data.train["meal_id"].astype(int))
val_meals = set(data.val["meal_id"].astype(int))
test_meals = set(data.test["meal_id"].astype(int))
all_meals = set(range(n_meals))

print("Total meal IDs:", len(all_meals))
print("Meals in training set:", len(train_meals))
print("Meals in validation set:", len(val_meals))
print("Meals in test set:", len(test_meals))

print("\nMeals absent from training set:", len(all_meals - train_meals))
print("Cold-start meals in validation:", len(val_meals - train_meals))
print("Cold-start meals in test:", len(test_meals - train_meals))

print("\nCold-start meal IDs:", sorted(all_meals - train_meals))

Total meal IDs: 3817
Meals in training set: 3810
Meals in validation set: 2332
Meals in test set: 2347

Meals absent from training set: 7
Cold-start meals in validation: 6
Cold-start meals in test: 3

Cold-start meal IDs: [734, 1672, 3194, 3431, 3608, 3731, 3774]


In [ ]:
#Step 23: Standardize the evaluation candidate set
def evaluate_topk_known_items(
    score_matrix,
    train,
    val,
    health_df,
    known_meals,
    k=10
):
    n_users, n_meals = score_matrix.shape

    seen = (
        train.groupby("user_id")["meal_id"]
        .apply(lambda x: set(x.astype(int)))
        .to_dict()
    )
    truth = (
        val.groupby("user_id")["meal_id"]
        .apply(lambda x: set(x.astype(int)))
        .to_dict()
    )
    health_map = health_df.set_index("meal_id")["health_score"].to_dict()

    recalls, precisions, ndcgs, health_values = [], [], [], []

    known_meals = np.array(sorted(known_meals), dtype=int)

    for user_id, relevant in truth.items():
        user_id = int(user_id)
        relevant = set(map(int, relevant))

        # Exclude meals absent from the training set and meals already seen
        candidates = known_meals[
            ~np.isin(known_meals, list(seen.get(user_id, set())))
        ]

        if len(candidates) == 0:
            continue

        scores = score_matrix[user_id, candidates]
        top = candidates[np.argsort(-scores)[:k]]

        hits = np.array([int(m in relevant) for m in top])
        recalls.append(
            hits.sum() / len(relevant) if relevant else 0.0
        )
        precisions.append(hits.sum() / k)

        discounts = 1.0 / np.log2(np.arange(2, len(top) + 2))
        dcg = np.sum(hits * discounts)
        ideal_hits = min(len(relevant), k)
        idcg = (
            np.sum(1.0 / np.log2(np.arange(2, ideal_hits + 2)))
            if ideal_hits > 0 else 0.0
        )
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

        meal_health = [
            health_map.get(int(m), np.nan) for m in top
        ]
        health_values.append(np.nanmean(meal_health))

    return {
        f"Recall@{k}": np.mean(recalls),
        f"Precision@{k}": np.mean(precisions),
        f"NDCG@{k}": np.mean(ndcgs),
        f"AvgHealth@{k}": np.nanmean(health_values)
    }

In [ ]:
#Step 24: Re-evaluate Popularity
known_meals = set(data.train["meal_id"].astype(int))

pop_test_metrics_fair = evaluate_topk_known_items(
    score_matrix=pop_scores,
    train=data.train,
    val=data.test,
    health_df=health_df,
    known_meals=known_meals,
    k=10
)

print("Popularity Test Results (Known Items Only)")
for metric, value in pop_test_metrics_fair.items():
    print(f"{metric}: {value:.6f}")

Popularity Test Results (Known Items Only)
Recall@10: 0.022726
Precision@10: 0.006912
NDCG@10: 0.013993
AvgHealth@10: 0.331079


In [ ]:
#Step 25: Re-evaluate both BPR models
known_meals = set(data.train["meal_id"].astype(int))

fair_test_rows = []

for seed in [42, 123, 2026]:
    for lam in [0.0, 0.025]:
        print(f"Training: seed={seed}, lambda_health={lam}")

        model = train_bpr(
            train=data.train,
            n_users=n_users,
            n_meals=n_meals,
            health_scores=health_scores,
            lambda_health=lam,
            factors=64,
            epochs=20,
            batch_size=2048,
            lr=1e-3,
            reg=1e-4,
            seed=seed
        )

        scores = get_score_matrix(
            model=model,
            n_users=n_users,
            n_meals=n_meals
        )

        metrics = evaluate_topk_known_items(
            score_matrix=scores,
            train=data.train,
            val=data.test,
            health_df=health_df,
            known_meals=known_meals,
            k=10
        )

        fair_test_rows.append({
            "seed": seed,
            "lambda_health": lam,
            **metrics
        })

fair_test_df = pd.DataFrame(fair_test_rows)

print("\nBPR Test Results (Known Items Only)")
print(fair_test_df.round(6).to_string(index=False))

print("\nMean and Standard Deviation")
fair_test_summary = fair_test_df.groupby("lambda_health")[
    ["Recall@10", "Precision@10", "NDCG@10", "AvgHealth@10"]
].agg(["mean", "std"])

print(fair_test_summary.round(6))

Training: seed=42, lambda_health=0.0
Epoch 01/20 loss=0.6933
Epoch 02/20 loss=0.6905
Epoch 03/20 loss=0.6882
Epoch 04/20 loss=0.6857
Epoch 05/20 loss=0.6830
Epoch 06/20 loss=0.6797
Epoch 07/20 loss=0.6758
Epoch 08/20 loss=0.6712
Epoch 09/20 loss=0.6656
Epoch 10/20 loss=0.6589
Epoch 11/20 loss=0.6509
Epoch 12/20 loss=0.6416
Epoch 13/20 loss=0.6305
Epoch 14/20 loss=0.6181
Epoch 15/20 loss=0.6039
Epoch 16/20 loss=0.5875
Epoch 17/20 loss=0.5709
Epoch 18/20 loss=0.5526
Epoch 19/20 loss=0.5343
Epoch 20/20 loss=0.5145
Training: seed=42, lambda_health=0.025
Epoch 01/20 loss=0.6934
Epoch 02/20 loss=0.6906
Epoch 03/20 loss=0.6883
Epoch 04/20 loss=0.6858
Epoch 05/20 loss=0.6831
Epoch 06/20 loss=0.6798
Epoch 07/20 loss=0.6759
Epoch 08/20 loss=0.6713
Epoch 09/20 loss=0.6657
Epoch 10/20 loss=0.6590
Epoch 11/20 loss=0.6510
Epoch 12/20 loss=0.6417
Epoch 13/20 loss=0.6305
Epoch 14/20 loss=0.6182
Epoch 15/20 loss=0.6040
Epoch 16/20 loss=0.5876
Epoch 17/20 loss=0.5710
Epoch 18/20 loss=0.5526
Epoch 19/20 

In [ ]:
def evaluate_topk_known_items(
    score_matrix, train, val, health_df, known_meals, k=10
):
    known_meals = set(map(int, known_meals))

    seen = (
        train.groupby("user_id")["meal_id"]
        .apply(lambda x: set(map(int, x)))
        .to_dict()
    )
    truth = (
        val.groupby("user_id")["meal_id"]
        .apply(lambda x: set(map(int, x)))
        .to_dict()
    )
    health_map = health_df.set_index("meal_id")["health_score"].to_dict()

    recalls, precisions, ndcgs, health_values = [], [], [], []

    for user_id, relevant_all in truth.items():
        user_id = int(user_id)

        # Only evaluate relevant meals available to the models
        relevant = set(map(int, relevant_all)) & known_meals
        if not relevant:
            continue

        candidates = sorted(known_meals - seen.get(user_id, set()))
        if not candidates:
            continue

        scores = score_matrix[user_id, candidates]
        top = np.asarray(candidates)[np.argsort(-scores)[:k]]

        hits = np.array([int(m in relevant) for m in top])
        recalls.append(hits.sum() / len(relevant))
        precisions.append(hits.sum() / k)

        discounts = 1.0 / np.log2(np.arange(2, len(top) + 2))
        dcg = np.sum(hits * discounts)
        ideal_hits = min(len(relevant), k)
        idcg = np.sum(
            1.0 / np.log2(np.arange(2, ideal_hits + 2))
        )
        ndcgs.append(dcg / idcg)

        meal_health = [
            health_map.get(int(m), np.nan) for m in top
        ]
        health_values.append(np.nanmean(meal_health))

    return {
        f"Recall@{k}": np.mean(recalls),
        f"Precision@{k}": np.mean(precisions),
        f"NDCG@{k}": np.mean(ndcgs),
        f"AvgHealth@{k}": np.nanmean(health_values)
    }

In [ ]:
#Step 26: Re-evaluate both BPR models
known_meals = set(data.train["meal_id"].astype(int))

pop_test_metrics_fair = evaluate_topk_known_items(
    score_matrix=pop_scores,
    train=data.train,
    val=data.test,
    health_df=health_df,
    known_meals=known_meals,
    k=10
)

print("Popularity Test Results (Corrected)")
for metric, value in pop_test_metrics_fair.items():
    print(f"{metric}: {value:.6f}")

Popularity Test Results (Corrected)
Recall@10: 0.022729
Precision@10: 0.006912
NDCG@10: 0.013993
AvgHealth@10: 0.331079


In [ ]:
#Step 27: Retrain and evaluate the 6 BPR model variants
known_meals = set(data.train["meal_id"].astype(int))

fair_test_rows = []

for seed in [42, 123, 2026]:
    for lam in [0.0, 0.025]:
        print(f"Training: seed={seed}, lambda_health={lam}")

        model = train_bpr(
            train=data.train,
            n_users=n_users,
            n_meals=n_meals,
            health_scores=health_scores,
            lambda_health=lam,
            factors=64,
            epochs=20,
            batch_size=2048,
            lr=1e-3,
            reg=1e-4,
            seed=seed
        )

        scores = get_score_matrix(
            model=model,
            n_users=n_users,
            n_meals=n_meals
        )

        metrics = evaluate_topk_known_items(
            score_matrix=scores,
            train=data.train,
            val=data.test,
            health_df=health_df,
            known_meals=known_meals,
            k=10
        )

        fair_test_rows.append({
            "seed": seed,
            "lambda_health": lam,
            **metrics
        })

fair_test_df = pd.DataFrame(fair_test_rows)

print("\nCorrected BPR Test Results")
print(fair_test_df.round(6).to_string(index=False))

print("\nMean and Standard Deviation")
fair_test_summary = fair_test_df.groupby("lambda_health")[
    ["Recall@10", "Precision@10", "NDCG@10", "AvgHealth@10"]
].agg(["mean", "std"])

print(fair_test_summary.round(6))

Training: seed=42, lambda_health=0.0
Epoch 01/20 loss=0.6933
Epoch 02/20 loss=0.6905
Epoch 03/20 loss=0.6882
Epoch 04/20 loss=0.6857
Epoch 05/20 loss=0.6830
Epoch 06/20 loss=0.6797
Epoch 07/20 loss=0.6758
Epoch 08/20 loss=0.6712
Epoch 09/20 loss=0.6656
Epoch 10/20 loss=0.6589
Epoch 11/20 loss=0.6509
Epoch 12/20 loss=0.6416
Epoch 13/20 loss=0.6305
Epoch 14/20 loss=0.6181
Epoch 15/20 loss=0.6039
Epoch 16/20 loss=0.5875
Epoch 17/20 loss=0.5709
Epoch 18/20 loss=0.5526
Epoch 19/20 loss=0.5343
Epoch 20/20 loss=0.5145
Training: seed=42, lambda_health=0.025
Epoch 01/20 loss=0.6934
Epoch 02/20 loss=0.6906
Epoch 03/20 loss=0.6883
Epoch 04/20 loss=0.6858
Epoch 05/20 loss=0.6831
Epoch 06/20 loss=0.6798
Epoch 07/20 loss=0.6759
Epoch 08/20 loss=0.6713
Epoch 09/20 loss=0.6657
Epoch 10/20 loss=0.6590
Epoch 11/20 loss=0.6510
Epoch 12/20 loss=0.6417
Epoch 13/20 loss=0.6305
Epoch 14/20 loss=0.6182
Epoch 15/20 loss=0.6040
Epoch 16/20 loss=0.5876
Epoch 17/20 loss=0.5710
Epoch 18/20 loss=0.5526
Epoch 19/20 

In [ ]:
# Step 27.1: train the model again using new parameters
# Validation-only lambda search
# Run this before Step 28.

lambda_grid = [0.0, 0.025, 0.05, 0.075, 0.1, 0.15, 0.2]
EPOCHS = 20
SEED = 42
K = 10
MAX_NDCG_DROP = 3.0

known_meals = set(data.train["meal_id"].astype(int))

health_map = health_df.set_index("meal_id")["health_score"].to_dict()

user_history_health = (
    data.train.assign(
        health_score=data.train["meal_id"].map(health_map)
    )
    .groupby("user_id")["health_score"]
    .mean()
)

history_health_baseline = user_history_health.mean()

validation_rows = []

for lam in lambda_grid:
    print(f"\nTraining lambda={lam}, epochs={EPOCHS}")

    model = train_bpr(
        train=data.train,
        n_users=n_users,
        n_meals=n_meals,
        health_scores=health_scores,
        lambda_health=lam,
        factors=64,
        epochs=EPOCHS,
        batch_size=2048,
        lr=1e-3,
        reg=1e-4,
        seed=SEED
    )

    score_matrix = get_score_matrix(model, n_users, n_meals)

    metrics = evaluate_topk_known_items(
        score_matrix=score_matrix,
        train=data.train,
        val=data.val,
        health_df=health_df,
        known_meals=known_meals,
        k=K
    )

    validation_rows.append({
        "lambda_health": lam,
        **metrics
    })

validation_df = pd.DataFrame(validation_rows)

# Calculate NDCG drop relative to ordinary BPR-MF
baseline_ndcg = validation_df.loc[
    validation_df["lambda_health"] == 0.0, "NDCG@10"
].iloc[0]

validation_df["NDCG_drop_pct"] = (
    (baseline_ndcg - validation_df["NDCG@10"])
    / baseline_ndcg * 100
)

validation_df["Health_gain_vs_history"] = (
    validation_df["AvgHealth@10"] - history_health_baseline
)

# Selection rule: NDCG drop <= 3%, health gain > 0
eligible = validation_df[
    (validation_df["NDCG_drop_pct"] <= MAX_NDCG_DROP)
    & (validation_df["Health_gain_vs_history"] > 0)
].sort_values("lambda_health")

print(f"Historical health baseline: {history_health_baseline:.6f}")
print(f"Maximum permitted NDCG drop: {MAX_NDCG_DROP:.1f}%")
print("\nAll validation results:")
display(validation_df.sort_values("lambda_health").round(6))

if len(eligible) == 0:
    selected_lambda = None
    print("No lambda satisfies both validation constraints.")
else:
    selected_row = eligible.iloc[-1]
    selected_lambda = float(selected_row["lambda_health"])

    print(f"\nSelected lambda: {selected_lambda}")
    print(f"NDCG@10: {selected_row['NDCG@10']:.6f}")
    print(f"NDCG drop: {selected_row['NDCG_drop_pct']:.2f}%")
    print(f"AvgHealth@10: {selected_row['AvgHealth@10']:.6f}")
    print(
        "Health gain vs history: "
        f"{selected_row['Health_gain_vs_history']:.6f}"
    )



Training lambda=0.0, epochs=20
Epoch 01/20 loss=0.6933
Epoch 02/20 loss=0.6905
Epoch 03/20 loss=0.6882
Epoch 04/20 loss=0.6857
Epoch 05/20 loss=0.6830
Epoch 06/20 loss=0.6797
Epoch 07/20 loss=0.6758
Epoch 08/20 loss=0.6712
Epoch 09/20 loss=0.6656
Epoch 10/20 loss=0.6589
Epoch 11/20 loss=0.6509
Epoch 12/20 loss=0.6416
Epoch 13/20 loss=0.6305
Epoch 14/20 loss=0.6181
Epoch 15/20 loss=0.6039
Epoch 16/20 loss=0.5875
Epoch 17/20 loss=0.5709
Epoch 18/20 loss=0.5526
Epoch 19/20 loss=0.5343
Epoch 20/20 loss=0.5145

Training lambda=0.025, epochs=20
Epoch 01/20 loss=0.6934
Epoch 02/20 loss=0.6906
Epoch 03/20 loss=0.6883
Epoch 04/20 loss=0.6858
Epoch 05/20 loss=0.6831
Epoch 06/20 loss=0.6798
Epoch 07/20 loss=0.6759
Epoch 08/20 loss=0.6713
Epoch 09/20 loss=0.6657
Epoch 10/20 loss=0.6590
Epoch 11/20 loss=0.6510
Epoch 12/20 loss=0.6417
Epoch 13/20 loss=0.6305
Epoch 14/20 loss=0.6182
Epoch 15/20 loss=0.6040
Epoch 16/20 loss=0.5876
Epoch 17/20 loss=0.5710
Epoch 18/20 loss=0.5526
Epoch 19/20 loss=0.534

,lambda_health,Recall@10,Precision@10,NDCG@10,AvgHealth@10,NDCG_drop_pct,Health_gain_vs_history
0,0.000,0.043858,0.014513,0.033456,0.331833,0.000000,-0.013199
1,0.025,0.043128,0.014425,0.033471,0.337055,-0.042633,-0.007977
2,0.050,0.042341,0.014336,0.032875,0.342296,1.739033,-0.002737
3,0.075,0.041899,0.014248,0.032591,0.347595,2.587134,0.002563
4,0.100,0.041456,0.014071,0.032512,0.353971,2.821637,0.008938
5,0.150,0.041957,0.014248,0.032752,0.367240,2.105759,0.022208
6,0.200,0.042566,0.014425,0.032278,0.382475,3.521192,0.037443



Selected lambda: 0.15
NDCG@10: 0.032752
NDCG drop: 2.11%
AvgHealth@10: 0.367240
Health gain vs history: 0.022208


In [ ]:

# Step 27.2: Generate final test results for Step 28

if selected_lambda is None:
    raise ValueError("selected_lambda is None. Run validation selection first.")

test_lambdas = [0.0, float(selected_lambda)]
test_seeds = [42, 123, 2026]
test_rows = []

known_meals = set(data.train["meal_id"].astype(int))

for lam in test_lambdas:
    for seed in test_seeds:
        print(f"Testing lambda={lam}, seed={seed}")

        model = train_bpr(
            train=data.train,
            n_users=n_users,
            n_meals=n_meals,
            health_scores=health_scores,
            lambda_health=lam,
            factors=64,
            epochs=20,
            batch_size=2048,
            lr=1e-3,
            reg=1e-4,
            seed=seed
        )

        score_matrix = get_score_matrix(
            model, n_users, n_meals
        )

        metrics = evaluate_topk_known_items(
            score_matrix=score_matrix,
            train=data.train,
            val=data.test,
            health_df=health_df,
            known_meals=known_meals,
            k=10
        )

        test_rows.append({
            "seed": seed,
            "lambda_health": lam,
            **metrics
        })

# Create the test-results DataFrame required by Step 28
fair_test_df = pd.DataFrame(test_rows)

# Calculate mean and standard deviation across seeds
metric_cols = [
    "Recall@10",
    "Precision@10",
    "NDCG@10",
    "AvgHealth@10"
]

fair_test_summary = fair_test_df.groupby("lambda_health")[
    metric_cols
].agg(["mean", "std"])

print("\nTest results by seed:")
display(fair_test_df.round(6))

print("\nTest results summary:")
display(fair_test_summary.round(6))


Testing lambda=0.0, seed=42
Epoch 01/20 loss=0.6933
Epoch 02/20 loss=0.6905
Epoch 03/20 loss=0.6882
Epoch 04/20 loss=0.6857
Epoch 05/20 loss=0.6830
Epoch 06/20 loss=0.6797
Epoch 07/20 loss=0.6758
Epoch 08/20 loss=0.6712
Epoch 09/20 loss=0.6656
Epoch 10/20 loss=0.6589
Epoch 11/20 loss=0.6509
Epoch 12/20 loss=0.6416
Epoch 13/20 loss=0.6305
Epoch 14/20 loss=0.6181
Epoch 15/20 loss=0.6039
Epoch 16/20 loss=0.5875
Epoch 17/20 loss=0.5709
Epoch 18/20 loss=0.5526
Epoch 19/20 loss=0.5343
Epoch 20/20 loss=0.5145
Testing lambda=0.0, seed=123
Epoch 01/20 loss=0.6932
Epoch 02/20 loss=0.6907
Epoch 03/20 loss=0.6884
Epoch 04/20 loss=0.6859
Epoch 05/20 loss=0.6831
Epoch 06/20 loss=0.6802
Epoch 07/20 loss=0.6764
Epoch 08/20 loss=0.6718
Epoch 09/20 loss=0.6667
Epoch 10/20 loss=0.6603
Epoch 11/20 loss=0.6529
Epoch 12/20 loss=0.6439
Epoch 13/20 loss=0.6334
Epoch 14/20 loss=0.6216
Epoch 15/20 loss=0.6076
Epoch 16/20 loss=0.5927
Epoch 17/20 loss=0.5758
Epoch 18/20 loss=0.5585
Epoch 19/20 loss=0.5401
Epoch 2

,seed,lambda_health,Recall@10,Precision@10,NDCG@10,AvgHealth@10
0,42,0.00,0.047256,0.015530,0.031058,0.332216
1,123,0.00,0.049965,0.014093,0.034941,0.330194
2,2026,0.00,0.052695,0.014811,0.040631,0.333989
3,42,0.15,0.049072,0.015978,0.032056,0.367745
4,123,0.15,0.049030,0.014004,0.034495,0.365428
5,2026,0.15,0.054904,0.015530,0.040789,0.368509



Test results summary:


Recall@10           Precision@10             NDCG@10            \
                   mean       std         mean       std      mean       std   
lambda_health                                                                  
0.00           0.049972  0.002720     0.014811  0.000718  0.035543  0.004815   
0.15           0.051002  0.003379     0.015171  0.001035  0.035780  0.004506   

              AvgHealth@10            
                      mean       std  
lambda_health                         
0.00              0.332133  0.001899  
0.15              0.367227  0.001604

In [ ]:
#Step 28: Save the final revised results
# 1. Calculate mean metrics across the three BPR seeds
fair_test_mean = fair_test_df.groupby("lambda_health")[
    ["Recall@10", "Precision@10", "NDCG@10", "AvgHealth@10"]
].mean()

# 2. Build the final comparison table
final_fair_test_df = pd.DataFrame([
    {
        "Model": "Popularity",
        **pop_test_metrics_fair
    },
    {
        "Model": "BPR-MF (lambda=0)",
        **fair_test_mean.loc[0.0].to_dict()
    },
    {
        "Model": "Health-aware BPR (lambda=selected_lambda)",
        **fair_test_mean.loc[selected_lambda].to_dict()
    }
])

# 3. Save results
final_fair_test_df.to_csv(
    os.path.join(RESULTS_DIR, "final_test_comparison.csv"),
    index=False
)

fair_test_df.to_csv(
    os.path.join(RESULTS_DIR, "bpr_test_results_by_seed.csv"),
    index=False
)

fair_test_summary.to_csv(
    os.path.join(RESULTS_DIR, "bpr_test_mean_std.csv")
)

# 4. Print and verify
print("Final Test Comparison:")
print(final_fair_test_df.round(6).to_string(index=False))

print("\nSaved files:")
print([
    "final_test_comparison.csv",
    "bpr_test_results_by_seed.csv",
    "bpr_test_mean_std.csv"
])

Final Test Comparison:
                                    Model  Recall@10  Precision@10  NDCG@10  AvgHealth@10
                               Popularity   0.022729      0.006912 0.013993      0.331079
                        BPR-MF (lambda=0)   0.049972      0.014811 0.035543      0.332133
Health-aware BPR (lambda=selected_lambda)   0.051002      0.015171 0.035780      0.367227

Saved files:
['final_test_comparison.csv', 'bpr_test_results_by_seed.csv', 'bpr_test_mean_std.csv']


In [ ]:

# Step 29: Save the model weights
import os
import torch

MODEL_DIR = os.path.join(ROOT, "models", "checkpoints")
os.makedirs(MODEL_DIR, exist_ok=True)

# Verify the lambda selected using the validation set
if selected_lambda is None:
    raise ValueError(
        "selected_lambda is None. "
        "Please run the validation-only lambda search first."
    )

selected_lambda = float(selected_lambda)

# Train the final Health-aware BPR model using the selected parameters
final_model = train_bpr(
    train=data.train,
    n_users=n_users,
    n_meals=n_meals,
    health_scores=health_scores,
    lambda_health=selected_lambda,
    factors=64,
    epochs=20,
    batch_size=2048,
    lr=1e-3,
    reg=1e-4,
    seed=42
)

checkpoint_path = os.path.join(
    MODEL_DIR, "health_aware_bpr.pt"
)

torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "n_users": n_users,
        "n_meals": n_meals,
        "factors": 64,
        "lambda_health": selected_lambda,
        "epochs": 20,
        "batch_size": 2048,
        "lr": 1e-3,
        "reg": 1e-4,
        "seed": 42
    },
    checkpoint_path
)

print("Selected lambda:", selected_lambda)
print("Saved model checkpoint:", checkpoint_path)


Epoch 01/20 loss=0.6941
Epoch 02/20 loss=0.6912
Epoch 03/20 loss=0.6889
Epoch 04/20 loss=0.6863
Epoch 05/20 loss=0.6837
Epoch 06/20 loss=0.6804
Epoch 07/20 loss=0.6764
Epoch 08/20 loss=0.6719
Epoch 09/20 loss=0.6663
Epoch 10/20 loss=0.6596
Epoch 11/20 loss=0.6515
Epoch 12/20 loss=0.6422
Epoch 13/20 loss=0.6310
Epoch 14/20 loss=0.6188
Epoch 15/20 loss=0.6044
Epoch 16/20 loss=0.5880
Epoch 17/20 loss=0.5713
Epoch 18/20 loss=0.5530
Epoch 19/20 loss=0.5347
Epoch 20/20 loss=0.5148
Selected lambda: 0.15
Saved model checkpoint: /content/drive/MyDrive/HealthRank/models/checkpoints/health_aware_bpr.pt


In [ ]:
#Step 30: Run interface tests
import os
import sys

ROOT = "/content/drive/MyDrive/HealthRank"
os.chdir(ROOT)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.inference import load_recommender, recommend_for_user

# Load the trained model only once
recommender = load_recommender()

# Try recommending meals for user 0
recommendations = recommend_for_user(
    recommender,
    user_id=0,
    k=10
)

display(recommendations)


,meal_id,score,health_score,fsa,who
0,1392,0.348658,0.425000,8.666667,2.333333
1,2185,0.321580,0.441667,9.333333,3.000000
2,18,0.321333,0.308333,9.333333,1.666667
3,2584,0.314340,0.408333,9.333333,2.666667
4,886,0.313215,0.366667,9.000000,2.000000
5,1750,0.301117,0.475000,9.333333,3.333333
6,1826,0.297492,0.516667,8.333333,3.000000
7,856,0.296416,0.375000,9.333333,2.333333
8,38,0.294806,0.641667,6.666667,3.000000
9,2396,0.293888,0.608333,8.000000,3.666667


load_recommender() should only be called once when the program starts; do not repeatedly load the model for every user request, as this will increase latency.


In [ ]:
#Step 31: Allow team members to call it directly
from src.inference import load_recommender, recommend_for_user

recommender = load_recommender()

# Get the top 10 recommendations for user 25
result = recommend_for_user(recommender, user_id=25, k=10)

print(result)


   meal_id     score  health_score        fsa       who
0      791  0.399076      0.366667   9.000000  2.000000
1      173  0.387665      0.516667   8.333333  3.000000
2      718  0.376485      0.383333   9.666667  2.666667
3     2419  0.361921      0.675000   6.666667  3.333333
4     1115  0.350185      0.416667   8.333333  2.000000
5       33  0.349878      0.316667   9.666667  2.000000
6      228  0.346936      0.300000  10.333333  2.333333
7      376  0.343091      0.308333   9.333333  1.666667
8     1207  0.340232      0.391667   8.666667  2.000000
9      240  0.338819      0.300000   9.000000  1.333333


I have implemented a Python inference interface for the health-aware BPR recommendation model in `src/inference.py`.

The interface provides:
- `load_recommender()` to load the trained model and supporting data.
- `recommend_for_user(recommender, user_id, k=10)` to generate Top-K meal recommendations.

The output includes meal IDs, ranking scores, and health-related information such as `health_score`, `fsa`, and `who`.

The model checkpoint is saved at `models/checkpoints/health_aware_bpr.pt`. The model does not need to be retrained for each request.

Please let me know if you need the interface adapted for the main application or a REST API.
